# Mass Spectrometry Group Alignment (GALAXY)

Anji Deng, Yuyang Zhang, Qihuang Zhang\*

McGill University &mdash; corresponding author: <qihuang.zhang@mcgill.ca>

Reference: Deng A, Zhang Y, Zhang Q. *GALAXY: Group Alignment of Mass Spectrometry data for imaging and spatial metabolomics*. Manuscript under review, 2026.

This tutorial demonstrates how to use **GALAXY** to align two MALDI mass spectrometry datasets
(e.g., Week 2 and Week 5 from the atherosclerosis regression experiment) and prepare them
for downstream joint analyses such as spatial segmentation.

We assume you have:

- Two CSV files containing the spectra (one row per pixel/spot, one column per m/z bin)  
- Two CSV files containing the spatial coordinates for each pixel/spot  
- The `GalaxyPython` package installed (e.g. via `pip install -e .` from the repo root)

---

## 0. Setup and package imports

Loading Packages

In [1]:
import os,csv,random
import pandas as pd
import numpy as np
import scanpy as sc
import math

from skimage import io, color
import torch


In [2]:
from scanpy import read_10x_h5
import SpaGCN as spg
import matplotlib.pyplot as plt
import json


In [3]:
from tqdm import tqdm
import pickle

In [ ]:
import GalaxyPython as gx
gx.__version__

## 1. Load MALDI data and create AnnData objects

Set `DataDir` below to the local folder where you have placed the four CSV files for
this tutorial (two weeks &times; spectra + region-spots). The Maaike (atherosclerosis
regression) dataset is available from the data owners upon reasonable request &mdash;
see `CodeInPaper/Maaike/README.md` for details.

In [ ]:
# === CONFIGURE ME ===
DataDir = "data/Maaike"   # path to the Maaike Schuurman dataset CSVs
# ====================

weekpoint1 = "5 wk regression"
weekpoint2 = "2 wk regression"

groupshort = "pos"

if groupshort == "pos":
    group = "DHB pos"
else:
    group = "9AA neg"

In [7]:
DataDir

'data/Maaike'

In [8]:
weekpoint1

'5 wk regression'

In [9]:
group

'DHB pos'

In [10]:
MALDIdata1 = pd.read_csv("{DataDir}/{weekpoint} - {group} - All Spectra.csv".format(weekpoint = weekpoint1, DataDir = DataDir, group = group), sep =';') 
MALDIloc1 = pd.read_csv("{DataDir}/{weekpoint} - {group} - Region Spots.csv".format(weekpoint = weekpoint1, DataDir = DataDir, group = group), sep =';') 
MALDIdata2 = pd.read_csv("{DataDir}/{weekpoint} - {group} - All Spectra.csv".format(weekpoint = weekpoint2, DataDir = DataDir, group = group), sep =';') 
MALDIloc2 = pd.read_csv("{DataDir}/{weekpoint} - {group} - Region Spots.csv".format(weekpoint = weekpoint2, DataDir = DataDir, group = group), sep =';') 

In [11]:
mz_value1 = pd.DataFrame(list(MALDIdata1.columns[1:]), index = list(MALDIdata1.columns[1:])).astype('float')
mz_value1 = mz_value1.rename(columns = {0:"m/z"})

mz_value2 = pd.DataFrame(list(MALDIdata2.columns[1:]), index = list(MALDIdata2.columns[1:])).astype('float')
mz_value2 = mz_value2.rename(columns = {0:"m/z"})

In [12]:
mz_value1

,m/z
10,10.000000
10.126314163208,10.126314
10.252628326416,10.252628
10.378943443298,10.378943
10.505257606506,10.505258
...,...
1019.8834228516,1019.883423
1020.0097045898,1020.009705
1020.1360473633,1020.136047
1020.2623291016,1020.262329


In [15]:
np.diff(mz_value1["m/z"])

array([0.12631416, 0.12631416, 0.12631512, ..., 0.12634277, 0.12628174,
       0.12634277])

In [ ]:
mz_value2

Create AnnData object for each MSI data

In [35]:
MALDIdataAnn1 = sc.AnnData(X = MALDIdata1.iloc[:,1:], var = mz_value1, obs = MALDIloc1)
MALDIdataAnn2 = sc.AnnData(X = MALDIdata2.iloc[:,1:], var = mz_value2, obs = MALDIloc2)

/tmp/ipykernel_258932/895207559.py:1: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  MALDIdataAnn1 = sc.AnnData(X = MALDIdata1.iloc[:,1:], var = mz_value1, obs = MALDIloc1)
/home/calcium/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/tmp/ipykernel_258932/895207559.py:2: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  MALDIdataAnn2 = sc.AnnData(X = MALDIdata2.iloc[:,1:], var = mz_value2, obs = MALDIloc2)


In [36]:
MALDIdataAnn1.obs["x"]

0          -5.817526
1           4.182474
2           4.182474
3           4.182474
4           4.182474
            ...     
18671    1694.182495
18672    1694.182495
18673    1694.182495
18674    1694.182495
18675    1694.182495
Name: x, Length: 18676, dtype: float64

## 2. Coordinate formatting and intensity normalization

GALAXY expects integer coordinates and benefits from per-spot normalization.

Transform the coordinates into integer, normalize the intensity value for each spot.

In [38]:
MALDIdataAnn1.obs = MALDIdataAnn1.obs.astype(int)
sc.pp.normalize_per_cell(MALDIdataAnn1)

In [39]:
MALDIdataAnn2.obs = MALDIdataAnn2.obs.astype(int)
sc.pp.normalize_per_cell(MALDIdataAnn2)

## 3. Peak calling and peak grouping (Steps 1 & 2)

We use the `PeakCalling` class to identify peaks (Step 1 of the paper) and to
group adjacent peaks into joint peak groups (Step 2).

In [ ]:
PeakGroup = gx.PeakCalling(MALDIdataAnn1, MALDIdataAnn2)
PeakGroup

**Step 1 &mdash; Peak Calling.** Use `.peak_calling(threshold)` to retain the top
`threshold` quantile of peaks. We set the threshold to 0.9 (i.e. the 90% quantile,
&alpha; in the paper) here.

In [ ]:
PeakGroup.peak_calling(threshold=0.9)

**Step 2 &mdash; Peak Grouping.** Use `.peak_grouping(percentile)` to group adjacent
peaks into joint peak groups. `percentile` is the quantile of between-peak
distances used as the cut-point: if two peaks are farther apart than this cut-point,
they fall into separate groups; otherwise they are merged.

Larger `percentile` &rArr; larger groups.

In [ ]:
PeakGroup.peak_grouping(percentile=0.9)

`peak_grouping(percentile)` pools the called peak m/z values from **both** spectra
into a single sorted sequence and partitions that sequence into joint peak groups,
cutting wherever the distance between two adjacent pooled peaks exceeds the
`percentile` quantile of the adjacent-distance distribution. This is one joint
partition of the pooled peaks, not a merge of two separately computed partitions.

`clusterUnk` and `clusterRef` apply the same partition to each spectrum on its own.
They are provided for inspection and are not used by the alignment steps, which take
`jointcluster`. For the first region,

In [67]:
PeakGroup.clusterUnk[0]

[23.01037979126, 23.136693954468, 23.263008117676]

In [68]:
PeakGroup.clusterRef[0]

[23.157243728638, 23.281370162964, 23.405494689941]

pooling the peaks from both spectra over that same region and partitioning the
pooled sequence gives the joint peak group

In [69]:
PeakGroup.jointcluster[0]

[23.01037979126,
 23.136693954468,
 23.157243728638,
 23.263008117676,
 23.281370162964,
 23.405494689941]

Then we can use save_clusterresults to save the clusters into csv files.

In [ ]:
gx.save_clusterresults(PeakGroup.clusterUnk, "output/example/clusterUnkranges")
gx.save_clusterresults(PeakGroup.clusterRef, "output/example/clusterRefranges")
gx.save_clusterresults(PeakGroup.jointcluster, "output/example/joint_combine_ranges")

## 4. Peak group pairing and fine alignment (Steps 3 & 4)

Build an alignment object and run the remaining GALAXY steps: correlation matrix,
Peak Group Pairing (Step 3), and Fine Alignment Assessment (Step 4).

First, create an object combining the unknown and reference spectra as a preparation for alignment.


In [ ]:
ExactAlign = gx.AnnDataMALDI(MALDIdataAnn1, MALDIdataAnn2)

Calculate correlation matrix for each pair of groups (one from unknown and another from the reference spectrum).

In [72]:
ExactAlign.get_corr_peakgroup_refined(PeakGroup.jointcluster)

100%|██████████| 132/132 [00:06<00:00, 20.81it/s]


**Step 3 &mdash; Peak Group Pairing.** A greedy algorithm on the (distance-penalised) similarity matrix pairs unknown groups with reference groups.

In [ ]:
ExactAlign.peak_group_pairing()

**Step 4: Fine Alignment Assessment.** For each paired peak group, slide a window of &plusmn;4 grid points (four spacings of the m/z grid, not an m/z interval of 4) and pick the rigid integer-index shift that maximises the diagonal-mean Pearson correlation. A group already in the right place attains its maximum at the zero offset and is left unshifted.

In [ ]:
ExactAlign.fine_alignment_assessment(threshold=0.2, ignore=True)

Finally, `summarize()` produce the aligned mz values.

In [80]:
ExactAlign.summarize()